In [15]:
!pip install -q neurokit2

In [16]:
"""
DỰ ÁN NGHIÊN CỨU IEEE: Nhận diện Căng thẳng Đáng tin cậy từ Cảm biến Cổ tay WESAD
Module: Tiền xử lý tín hiệu (BVP, EDA, TEMP, ACC) và Trích xuất Đặc trưng Cửa sổ Trượt (60s, slide 5s)
"""

import os
import pickle
import warnings
import numpy as np
import pandas as pd
from scipy import stats, signal
from scipy.interpolate import interp1d
try:
    from scipy.integrate import trapezoid as trapz_compat
except ImportError:
    from numpy import trapz as trapz_compat

import neurokit2 as nk
from tqdm import tqdm

warnings.filterwarnings('ignore')

# 1. CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ THỰC NGHIỆM

In [17]:
BASE_WESAD_DIR = "/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD"

SUBJECT_LIST = ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 
                'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17']

WINDOW_SIZE_SEC = 60      # Cửa sổ W = 60 giây
SLIDE_STEP_SEC  = 5       # Bước trượt Δt = 5 giây
PURITY_THRESHOLD = 0.80   # Độ tinh khiết nhãn tối thiểu

# Tần số lấy mẫu cảm biến (Hz)
FS = {
    'BVP': 64,
    'EDA': 4,
    'TEMP': 4,
    'ACC': 32,
    'LABEL': 700
}

# 1: Baseline (Không Stress) -> 0; 2: Stress -> 1
LABEL_MAP = {1: 0, 2: 1}

# 2. HÀM TÍNH TOÁN ĐẶC TRƯNG TỪNG CẢM BIẾN

In [18]:
def compute_robust_frequency_hrv(valid_rri_ms, fs_interp=4.0):
    """
    Nội suy chuỗi RRI không đều về lưới đều 4Hz và tính phổ Welch.
    Tránh hiện tượng NaN khi gọi thư viện ngoài trên cửa sổ ngắn 60s.
    """
    # Nếu cửa sổ 60s có dưới 10 nhịp đập, dữ liệu bị nhiễu cơ học quá nặng -> gán NaN
    if len(valid_rri_ms) < 10:
        return np.nan, np.nan, np.nan

    # 1. Xây dựng trục thời gian tích lũy cho từng nhịp (giây)
    rri_seconds = valid_rri_ms / 1000.0
    rri_time = np.cumsum(rri_seconds)
    rri_time = rri_time - rri_time[0]
    total_time = rri_time[-1]

    if total_time <= 10.0:
        return np.nan, np.nan, np.nan

    # 2. Nội suy bậc 3 (Cubic Spline) về tần số lấy mẫu đều 4Hz
    time_regular = np.arange(0, total_time, 1.0 / fs_interp)
    if len(time_regular) < 32:
        return np.nan, np.nan, np.nan

    try:
        interp_fn = interp1d(rri_time, valid_rri_ms, kind='cubic', fill_value='extrapolate')
        rri_uniform = interp_fn(time_regular)
        
        # Khử thành phần một chiều (Detrending)
        rri_detrended = signal.detrend(rri_uniform)

        # 3. Tính mật độ phổ công suất PSD bằng phương pháp Welch
        nperseg = min(len(rri_detrended), 64) # Cửa sổ phân đoạn 16 giây tại 4Hz
        freqs, psd = signal.welch(rri_detrended, fs=fs_interp, nperseg=nperseg, noverlap=nperseg//2)

        # 4. Tích phân năng lượng trên dải LF (0.04 - 0.15 Hz) và HF (0.15 - 0.40 Hz)
        lf_mask = (freqs >= 0.04) & (freqs < 0.15)
        hf_mask = (freqs >= 0.15) & (freqs <= 0.40)

        lf_power = float(trapz_compat(psd[lf_mask], freqs[lf_mask])) if np.any(lf_mask) else 0.0
        hf_power = float(trapz_compat(psd[hf_mask], freqs[hf_mask])) if np.any(hf_mask) else 0.0

        # Tránh chia cho 0
        if hf_power <= 1e-8:
            lf_hf = np.nan
        else:
            lf_hf = lf_power / hf_power

        return lf_power, hf_power, lf_hf
    except Exception:
        return np.nan, np.nan, np.nan

# 3. CÁC HÀM TRÍCH XUẤT ĐẶC TRƯNG MIỀN THỜI GIAN & TẦN SỐ

In [19]:
def extract_acc_features(acc_win):
    mag = np.linalg.norm(acc_win, axis=1)
    
    # Năng lượng phổ gia tốc
    freqs, psd = signal.welch(mag, fs=FS['ACC'], nperseg=min(len(mag), 256))
    spectral_energy = float(np.sum(psd))
    
    # Zero Crossing Rate
    mag_centered = mag - np.mean(mag)
    zcr = float(np.mean(np.diff(np.sign(mag_centered)) != 0)) if len(mag_centered) > 1 else 0.0

    return {
        'acc_mag_mean': float(np.mean(mag)),
        'acc_mag_std': float(np.std(mag)),
        'acc_mag_var': float(np.var(mag)),  # Thước đo rung lắc xác định FAR_motion
        'acc_mag_max': float(np.max(mag)),
        'acc_mag_energy': spectral_energy,
        'acc_mag_zcr': zcr
    }

def extract_temp_features(temp_win):
    slope = 0.0
    if len(temp_win) > 1:
        x = np.arange(len(temp_win))
        slope, _, _, _, _ = stats.linregress(x, temp_win)
    return {
        'temp_mean': float(np.mean(temp_win)),
        'temp_std': float(np.std(temp_win)),
        'temp_min': float(np.min(temp_win)),
        'temp_max': float(np.max(temp_win)),
        'temp_slope': float(slope)
    }

def extract_eda_features(tonic_win, phasic_win, peaks_win):
    return {
        'eda_tonic_mean': float(np.mean(tonic_win)),
        'eda_tonic_std': float(np.std(tonic_win)),
        'eda_phasic_mean': float(np.mean(phasic_win)),
        'eda_phasic_std': float(np.std(phasic_win)),
        'eda_phasic_max': float(np.max(phasic_win)) if len(phasic_win) > 0 else 0.0,
        'eda_phasic_energy': float(np.sum(phasic_win ** 2)),
        'eda_scr_peaks_count': int(np.sum(peaks_win))
    }

def extract_hrv_features(peak_indices_in_win, fs=64):
    res = {
        'hrv_mean_hr': np.nan, 'hrv_sdnn': np.nan,
        'hrv_rmssd': np.nan, 'hrv_pnn50': np.nan,
        'hrv_lf': np.nan, 'hrv_hf': np.nan, 'hrv_lf_hf': np.nan
    }
    
    if len(peak_indices_in_win) < 4:
        return res

    # Khoảng cách RRI tính theo mili-giây
    ibi_ms = np.diff(peak_indices_in_win) / fs * 1000.0
    
    # Lọc ngưỡng sinh lý: 300ms (200 bpm) đến 1800ms (33 bpm)
    valid_ibi = ibi_ms[(ibi_ms >= 300) & (ibi_ms <= 1800)]
    if len(valid_ibi) < 4:
        return res

    # Đặc trưng miền thời gian
    res['hrv_mean_hr'] = float(60000.0 / np.mean(valid_ibi))
    res['hrv_sdnn'] = float(np.std(valid_ibi, ddof=1))
    
    diff_ibi = np.diff(valid_ibi)
    res['hrv_rmssd'] = float(np.sqrt(np.mean(diff_ibi ** 2)))
    res['hrv_pnn50'] = float((np.sum(np.abs(diff_ibi) > 50) / len(diff_ibi)) * 100.0)

    # Đặc trưng miền tần số cải tiến
    lf, hf, lf_hf = compute_robust_frequency_hrv(valid_ibi)
    res['hrv_lf'] = lf
    res['hrv_hf'] = hf
    res['hrv_lf_hf'] = lf_hf

    return res

# 4. TIỀN XỬ LÝ VÀ PHÂN ĐOẠN CỬA SỔ THEO ĐỐI TƯỢNG

In [20]:
def process_single_subject(subject_id, file_path):
    with open(file_path, 'rb') as f:
        subject_data = pickle.load(f, encoding='latin1')
    
    wrist = subject_data['signal']['wrist']
    raw_bvp  = wrist['BVP'].flatten()
    raw_eda  = wrist['EDA'].flatten()
    raw_temp = wrist['TEMP'].flatten()
    raw_acc  = wrist['ACC']  # Shape (N, 3)
    raw_lbl  = subject_data['label'].flatten()

    duration_sec = int(min(
        len(raw_bvp) / FS['BVP'],
        len(raw_eda) / FS['EDA'],
        len(raw_temp) / FS['TEMP'],
        len(raw_acc) / FS['ACC'],
        len(raw_lbl) / FS['LABEL']
    ))

    # --- Tiền xử lý toàn bộ phiên ghi ---
    # 1. BVP: Lọc nhiễu PPG và tìm đỉnh
    bvp_clean = nk.ppg_clean(raw_bvp, sampling_rate=FS['BVP'])
    peaks_dict = nk.ppg_findpeaks(bvp_clean, sampling_rate=FS['BVP'])
    bvp_peak_indices = np.array(peaks_dict['PPG_Peaks'])

    # 2. EDA: Lọc nhiễu và tách Tonic/Phasic
    eda_cleaned = nk.eda_clean(raw_eda, sampling_rate=FS['EDA'])
    eda_decomp = nk.eda_phasic(eda_cleaned, sampling_rate=FS['EDA'])
    tonic_signal = eda_decomp['EDA_Tonic'].values
    phasic_signal = eda_decomp['EDA_Phasic'].values
    eda_peaks_info = nk.eda_peaks(phasic_signal, sampling_rate=FS['EDA'])
    scr_peak_indicators = eda_peaks_info[0]['SCR_Peaks'].values

    # --- Cắt cửa sổ trượt ---
    extracted_rows = []
    for t_start in range(0, duration_sec - WINDOW_SIZE_SEC + 1, SLIDE_STEP_SEC):
        t_end = t_start + WINDOW_SIZE_SEC

        idx_lbl_0 = int(t_start * FS['LABEL'])
        idx_lbl_1 = int(t_end * FS['LABEL'])
        lbl_chunk = raw_lbl[idx_lbl_0:idx_lbl_1]

        mode_val, mode_count = stats.mode(lbl_chunk, keepdims=False)
        purity = mode_count / len(lbl_chunk)

        if purity < PURITY_THRESHOLD or mode_val not in LABEL_MAP:
            continue

        label_binary = LABEL_MAP[mode_val]

        acc_chunk  = raw_acc[int(t_start * FS['ACC']) : int(t_end * FS['ACC'])]
        temp_chunk = raw_temp[int(t_start * FS['TEMP']) : int(t_end * FS['TEMP'])]
        
        tonic_chunk  = tonic_signal[int(t_start * FS['EDA']) : int(t_end * FS['EDA'])]
        phasic_chunk = phasic_signal[int(t_start * FS['EDA']) : int(t_end * FS['EDA'])]
        scr_chunk    = scr_peak_indicators[int(t_start * FS['EDA']) : int(t_end * FS['EDA'])]

        idx_bvp_0 = int(t_start * FS['BVP'])
        idx_bvp_1 = int(t_end * FS['BVP'])
        peaks_in_window = bvp_peak_indices[(bvp_peak_indices >= idx_bvp_0) & (bvp_peak_indices < idx_bvp_1)] - idx_bvp_0

        # Trích xuất vector đặc trưng
        f_acc = extract_acc_features(acc_chunk)
        f_temp = extract_temp_features(temp_chunk)
        f_eda = extract_eda_features(tonic_chunk, phasic_chunk, scr_chunk)
        f_hrv = extract_hrv_features(peaks_in_window, fs=FS['BVP'])

        sample_row = {
            'subject_id': subject_id,
            'time_start_sec': t_start,
            'time_end_sec': t_end,
            'label': label_binary,
            **f_acc,
            **f_temp,
            **f_eda,
            **f_hrv
        }
        extracted_rows.append(sample_row)

    return pd.DataFrame(extracted_rows)

# 5. ĐIỀU PHỐI VÀ LƯU DATASET CHUẨN

In [21]:
def main():
    print("=== TIẾN HÀNH TRÍCH XUẤT ĐẶC TRƯNG WESAD (ĐÃ FIX TẦN SỐ HRV) ===")
    all_dfs = []

    for sid in tqdm(SUBJECT_LIST, desc="Xử lý 15 Subjects"):
        pkl_path = os.path.join(BASE_WESAD_DIR, sid, f"{sid}.pkl")
        if not os.path.exists(pkl_path):
            print(f"[!] Bỏ qua {sid}: Không thấy file {pkl_path}")
            continue
        df_sub = process_single_subject(sid, pkl_path)
        all_dfs.append(df_sub)

    df_dataset = pd.concat(all_dfs, ignore_index=True)

    print("\n[+] Trích xuất hoàn tất. Đang thực hiện Imputation chuẩn học thuật...")
    feature_cols = [c for c in df_dataset.columns if c not in ['subject_id', 'time_start_sec', 'time_end_sec', 'label']]
    
    # 1. Điền median theo từng Subject (bảo toàn đặc tính cá thể)
    df_dataset[feature_cols] = df_dataset.groupby('subject_id')[feature_cols].transform(lambda g: g.fillna(g.median()))
    
    # 2. Điền median toàn cục cho các trường hợp còn sót lại (nếu có)
    df_dataset[feature_cols] = df_dataset[feature_cols].fillna(df_dataset[feature_cols].median())

    output_filename = "wesad_wrist_features.csv"
    df_dataset.to_csv(output_filename, index=False)

    print("\n================ BÁO CÁO NGHIÊN CỨU TỔNG KẾT ================")
    print(f"[+] File xuất: {output_filename}")
    print(f"[+] Kích thước ma trận dữ liệu: {df_dataset.shape}")
    print(f"[+] Tổng số giá trị NaN còn sót: {df_dataset.isnull().sum().sum()} (Kỳ vọng: 0)")
    print(f"[+] Kiểm tra đặc trưng HRV Frequency:")
    print(f"    - Mean LF: {df_dataset['hrv_lf'].mean():.4f}")
    print(f"    - Mean HF: {df_dataset['hrv_hf'].mean():.4f}")
    print(f"    - Mean LF/HF Ratio: {df_dataset['hrv_lf_hf'].mean():.4f}")
    print(f"[+] Phân phối nhãn thực nghiệm:")
    print(df_dataset['label'].value_counts(normalize=True).apply(lambda x: f"{x*100:.2f}%"))
    print("=============================================================")

if __name__ == "__main__":
    main()

=== TIẾN HÀNH TRÍCH XUẤT ĐẶC TRƯNG WESAD (ĐÃ FIX TẦN SỐ HRV) ===


Xử lý 15 Subjects: 100%|██████████| 15/15 [01:29<00:00,  5.95s/it]



[+] Trích xuất hoàn tất. Đang thực hiện Imputation chuẩn học thuật...

================ BÁO CÁO NGHIÊN CỨU TỔNG KẾT ================
[+] File xuất: wesad_wrist_features.csv
[+] Kích thước ma trận dữ liệu: (5294, 29)
[+] Tổng số giá trị NaN còn sót: 0 (Kỳ vọng: 0)
[+] Kiểm tra đặc trưng HRV Frequency:
    - Mean LF: 8453.9271
    - Mean HF: 22984.4310
    - Mean LF/HF Ratio: 0.4216
[+] Phân phối nhãn thực nghiệm:
label
0    64.45%
1    35.55%
Name: proportion, dtype: object
